# 🎯 Notebook 5 — Orchestrator Deployment and Routing Smoke Tests

The orchestrator is the single entry point for all Product Finder requests.  
It reads the runtime discovery snapshot, applies governance rules, routes to the right specialists, and bundles the final answer.

## Dynamic routing logic
```
User query + governance context (persona, disclaimer_accepted)
        │
        ▼ [pf-orchestrator receives message]
  Intent classified by contextualizer call
        │
        ├── recommendation intent, low risk
        │       → product-intelligence → aligner
        │
        ├── compatibility intent, elevated risk
        │       ├── disclaimer NOT accepted → return DISCLAIMER_GATE response
        │       └── disclaimer accepted     → product-intelligence + compatibility → aligner
        │
        ├── sample_request intent
        │       ├── external_customer persona → sample-request agent
        │       └── other persona             → AUTH_DENIED response
        │
        └── out_of_domain → polite refusal with allowed topics
```

## Bundle structure returned to caller
```json
{
  "final_answer": "...",
  "agents_used": ["pf-contextualizer", "pf-product-intelligence", "pf-aligner"],
  "routing_decision": {"intent": "...", "risk_tier": "...", "persona": "..."},
  "confidence": 0.91,
  "governance_notices": [],
  "disclaimer_required": false
}
```

In [ ]:
import sys, json, pathlib, time as _t, uuid, subprocess, datetime, os, re
sys.path.insert(0, str(pathlib.Path("../../shared").resolve()))
import utils  # type: ignore

def run(cmd, ok="", fail=""):
    return utils.run(cmd, ok, fail)

def find_repo_root(start: pathlib.Path) -> pathlib.Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "bicep" / "infra" / "citadel-access-contracts" / "main.bicep").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing bicep/infra/citadel-access-contracts/main.bicep")

def azd_get(key: str) -> str:
    r = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Missing azd env value [{key}]: {(r.stderr or r.stdout).strip()}")
    return (r.stdout or "").strip()

def azd_get_optional(key: str, default: str = "") -> str:
    r = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if r.returncode != 0:
        return default
    return (r.stdout or "").strip() or default

def set_azd_env(key: str, value: str):
    r = subprocess.run(["azd", "env", "set", key, value], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Failed to persist azd env {key}: {(r.stderr or r.stdout).strip()}")

def next_version_tag(tag: str) -> str:
    match = re.fullmatch(r"v(\d+)", (tag or "").strip(), re.IGNORECASE)
    if not match:
        return "v1"
    return f"v{int(match.group(1)) + 1}"

def get_latest_v_tag_from_acr(repository: str) -> str:
    tags_out = run(
        f"az acr repository show-tags --name {ACR_NAME} --repository {repository} --output json",
        "",
        ""
    )
    if not tags_out.success:
        return ""

    tags = tags_out.json_data if isinstance(tags_out.json_data, list) else []
    latest = 0
    for tag in tags:
        if not isinstance(tag, str):
            continue
        match = re.fullmatch(r"v(\d+)", tag.strip(), re.IGNORECASE)
        if match:
            latest = max(latest, int(match.group(1)))

    return f"v{latest}" if latest > 0 else ""

# Runtime config from azd env (validation-style).
SUB_ID = azd_get("AZURE_SUBSCRIPTION_ID")
HUB_RG = azd_get("AZURE_RESOURCE_GROUP")
SPOKE_RG = azd_get("SPOKE_RESOURCE_GROUP")
FOUNDRY_ACCT = azd_get("SPOKE_AI_FOUNDRY_ACCOUNT_NAME")
SPOKE_PROJECT = azd_get("SPOKE_AI_FOUNDRY_PROJECT_NAME")
ACR_NAME = azd_get("SPOKE_ACR_NAME")
ACR_SERVER = azd_get("SPOKE_ACR_LOGIN_SERVER")
AZURE_LOCATION = azd_get("AZURE_LOCATION")

PF_MODEL_CONNECTION = azd_get_optional("PF_MODEL_CONNECTION", "Product-Finder-DEV-LLM")
PF_MODEL_DEPLOYMENT = azd_get_optional("PF_MODEL_DEPLOYMENT", "gpt-4.1")
PF_MODEL_FULL = azd_get_optional("PF_MODEL_FULL", f"{PF_MODEL_CONNECTION}/{PF_MODEL_DEPLOYMENT}")
ORCHESTRATOR_TAG_ENV_KEY = "PF_IMAGE_TAG_PF_ORCHESTRATOR"
PREVIOUS_IMAGE_TAG = get_latest_v_tag_from_acr("pf-orchestrator")
ORCHESTRATOR_IMAGE_TAG = PREVIOUS_IMAGE_TAG
FOUNDRY_EP = azd_get_optional("FOUNDRY_PROJECT_ENDPOINT", f"https://{FOUNDRY_ACCT}.services.ai.azure.com/api/projects/{SPOKE_PROJECT}")
SNAP_ID = azd_get_optional("PF_RUNTIME_SNAPSHOT_ID", "")
SNAP_CHKSUM = azd_get_optional("PF_RUNTIME_SNAPSHOT_CHECKSUM", "")
PF_SUB_KEY = azd_get_optional("PF_SUBSCRIPTION_KEY", "")
APIM_GATEWAY_URL = azd_get_optional("APIM_GATEWAY_URL", "").rstrip("/")

# Resolve APIM in hub RG.
apim_out = run(
    f"az resource list -g {HUB_RG} --resource-type Microsoft.ApiManagement/service -o json",
    "APIM query OK", "APIM query failed"
 )
if not apim_out.success or not apim_out.json_data:
    raise RuntimeError("No APIM resource found in hub resource group.")
APIM_NAME = apim_out.json_data[0]["name"]

ORCHESTRATOR_NAME = "pf-orchestrator"
AGENTS_BASE = pathlib.Path("../agents").resolve()

utils.print_info(f"Orchestrator:      {ORCHESTRATOR_NAME}")
utils.print_info(f"Model:             {PF_MODEL_FULL}")
utils.print_info(f"Current image tag: {PREVIOUS_IMAGE_TAG or '<none>'}")
utils.print_info(f"Next image tag:    {next_version_tag(PREVIOUS_IMAGE_TAG)}")
utils.print_info(f"Snapshot ID:       {SNAP_ID}")
utils.print_info(f"Snapshot chksum:   {SNAP_CHKSUM[:16]}...")

if not SNAP_ID:
    raise RuntimeError("No runtime snapshot found. Run Notebook 4 first.")

if not APIM_GATEWAY_URL:
    gw_out = run(
        f"az apim show -g {HUB_RG} -n {APIM_NAME} --query gatewayUrl -o tsv",
        "APIM gateway URL fetched", "Failed to fetch APIM gateway URL"
    )
    if not gw_out.success:
        raise RuntimeError("APIM gateway URL not found.")
    APIM_GATEWAY_URL = gw_out.stdout.strip().rstrip("/")

utils.print_info(f"APIM gateway:      {APIM_GATEWAY_URL}")
utils.print_info(f"Foundry endpoint:  {FOUNDRY_EP}")
utils.print_info(f"ACR:               {ACR_SERVER}")

In [ ]:
# ── 1️⃣  Write orchestrator source files ─────────────────────────────────────
orch_dir = AGENTS_BASE / ORCHESTRATOR_NAME
orch_dir.mkdir(exist_ok=True)

# Write the runtime snapshot into the orchestrator's source directory
# so it is bundled into the container image at build time
snap_src = pathlib.Path("../registry/runtime-snapshot.json").read_text()
(orch_dir / "runtime-snapshot.json").write_text(snap_src, encoding="utf-8")
utils.print_ok(f"runtime-snapshot.json copied into {orch_dir.name}/")

ORCHESTRATOR_MAIN = f'''import os, asyncio, json, uuid, requests
from typing import Annotated
from pathlib import Path
from pydantic import Field
from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from agent_framework_foundry_hosting import ResponsesHostServer
from azure.identity import DefaultAzureCredential

# ── Load runtime discovery snapshot ──────────────────────────────────────────
_SNAP_PATH = Path(os.environ.get("REGISTRY_SNAPSHOT_PATH", "runtime-snapshot.json"))
_SNAPSHOT = json.loads(_SNAP_PATH.read_text())
_AGENTS_INDEX = {{a["agent_name"]: a for a in _SNAPSHOT["agents"]}}

_APIM_GATEWAY_URL = os.environ["APIM_GATEWAY_URL"].rstrip("/")
_APIM_SUBSCRIPTION_KEY = os.environ["APIM_SUBSCRIPTION_KEY"]

def discover_agent(capability: str, persona: str, risk_tier: str) -> dict | None:
    """Return the first active agent that matches capability + persona + risk tier."""
    for agent in _SNAPSHOT["agents"]:
        if agent["status"] != "active":
            continue
        if capability not in agent.get("capabilities", []):
            continue
        if persona not in agent.get("allowed_personas", []):
            continue
        if risk_tier not in agent.get("risk_tiers_supported", []):
            continue
        return agent
    return None

def _extract_output_text(payload: object) -> str:
    if isinstance(payload, dict):
        if isinstance(payload.get("output_text"), str):
            return payload["output_text"]
        output = payload.get("output")
        if isinstance(output, list):
            chunks = []
            for item in output:
                if not isinstance(item, dict):
                    continue
                for c in item.get("content", []):
                    if isinstance(c, dict) and c.get("type") in ("output_text", "text") and c.get("text"):
                        chunks.append(c["text"])
            if chunks:
                    return "\n".join(chunks)
        return json.dumps(payload)
    if isinstance(payload, str):
        return payload
    return str(payload)

def _call_agent(agent: dict, message: str, persona: str) -> str:
    """Call a specialist through APIM gateway and return text output."""
    gateway_path = (agent.get("gateway_path") or "").strip()
    if not gateway_path:
        return json.dumps({{"error": "No gateway_path found for discovered agent", "agent": agent.get("agent_name")}})

    url = f"{{_APIM_GATEWAY_URL}}/{{gateway_path.lstrip('/')}}"
    headers = {{
        "Ocp-Apim-Subscription-Key": _APIM_SUBSCRIPTION_KEY,
        "x-user-persona": persona,
        "Content-Type": "application/json",
    }}
    payload = {{
        "input": message,
        "metadata": {{"conversation_id": str(uuid.uuid4()), "target_agent": agent.get("agent_name")}},
    }}
    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=90)
        if resp.status_code >= 400:
            return json.dumps({{
                "error": f"Gateway call failed: HTTP {{resp.status_code}}",
                "agent": agent.get("agent_name"),
                "gateway_path": gateway_path,
                "body": resp.text[:1200],
            }})
        try:
            return _extract_output_text(resp.json())
        except Exception:
            return resp.text
    except Exception as e:
        return json.dumps({{"error": str(e), "agent": agent.get("agent_name"), "gateway_path": gateway_path}})

# ── Orchestrator tools ────────────────────────────────────────────────────────
@tool(approval_mode="never_require")
def contextualize_query(
    user_query: Annotated[str, Field(description="Raw user query to contextualize")],
    persona: Annotated[str, Field(description="User persona: external_customer or internal_scientist")] = "external_customer",
    chat_history: Annotated[str, Field(description="Previous context or empty string")] = ""
) -> str:
    """Contextualize the user query: extract intent, entities, risk tier, and missing context."""
    agent = discover_agent("intent_extraction", persona, "low")
    if not agent:
        return json.dumps({{"error": "Contextualizer not available in registry"}})
    msg = user_query if not chat_history else f"{{user_query}}\\n\\nChat history: {{chat_history}}"
    return _call_agent(agent, msg, persona)

@tool(approval_mode="never_require")
def get_product_intelligence(
    contextualized_query: Annotated[str, Field(description="Refined query from contextualizer")],
    persona: Annotated[str, Field(description="User persona: external_customer or internal_scientist")] = "external_customer"
) -> str:
    """Search the product catalog and return matching product recommendations."""
    agent = discover_agent("product_recommendation", persona, "low")
    if not agent:
        return json.dumps({{"error": "Product intelligence agent not available in registry"}})
    return _call_agent(agent, contextualized_query, persona)

@tool(approval_mode="never_require")
def check_compatibility(
    product_a: Annotated[str, Field(description="First product name or ID")],
    product_b: Annotated[str, Field(description="Second product name or ID")],
    persona: Annotated[str, Field(description="User persona: external_customer or internal_scientist")] = "external_customer"
) -> str:
    """Check compatibility between two products. Returns verdict, confidence, and safety rationale."""
    agent = discover_agent("compatibility_check", persona, "elevated")
    if not agent:
        return json.dumps({{"error": "Compatibility agent not available in registry"}})
    return _call_agent(agent, f"Can I combine {{product_a}} with {{product_b}}?", persona)

@tool(approval_mode="never_require")
def align_response(
    draft_bundle: Annotated[str, Field(description="JSON string with all agent outputs to align and format")],
    persona: Annotated[str, Field(description="User persona: external_customer or internal_scientist")] = "external_customer"
) -> str:
    """Validate and reformat the bundled agent outputs into a clean final response."""
    agent = discover_agent("intent_alignment", persona, "low")
    if not agent:
        return json.dumps({{"error": "Aligner agent not available in registry"}})
    return _call_agent(agent, draft_bundle, persona)

@tool(approval_mode="never_require")
def process_sample_request(
    product_name: Annotated[str, Field(description="Product name the user wants a sample of")],
    persona: Annotated[str, Field(description="User persona: external_customer or internal_scientist")]
) -> str:
    """Process a product sample request for an authenticated external customer."""
    if persona != "external_customer":
        return json.dumps({{"error": "Sample requests are only available to external customers",
                            "auth_status": "not_authorized"}})
    agent = discover_agent("sample_request_processing", persona, "low")
    if not agent:
        return json.dumps({{"error": "Sample request agent not available in registry"}})
    msg = f"[GOVERNANCE CONTEXT]\\npersona: {{persona}}\\n\\nI would like a sample of {{product_name}}."
    return _call_agent(agent, msg, persona)

# ── Orchestrator agent ────────────────────────────────────────────────────────
ORCHESTRATOR_SYSTEM = """You are the Syensqo Product Finder Orchestrator.
You dynamically route user queries to the appropriate specialist agents and bundle their responses.

You receive messages in this format:
[GOVERNANCE CONTEXT]
persona: external_customer|internal_scientist
disclaimer_accepted: true|false

USER QUERY: <the user's question>

Routing rules you MUST follow:
1. ALWAYS call contextualize_query first to understand intent, entities, and risk_tier.
2. If intent=out_of_domain: respond politely explaining only product queries are supported.
3. If intent=recommendation:
   - Call get_product_intelligence with the contextualized_query
   - Call align_response with a JSON bundle of all outputs
4. If intent=compatibility AND risk_tier=elevated:
   - If disclaimer_accepted=false: STOP and return a DISCLAIMER_GATE response asking the user to accept before proceeding
   - If disclaimer_accepted=true: Call get_product_intelligence AND check_compatibility (use both product names from entities)
   - Then call align_response with the bundled outputs
5. If intent=sample_request: call process_sample_request with the product name and persona
6. If missing_context is non-empty: ask the user those clarifying questions before proceeding
7. When calling tools, ALWAYS pass the persona extracted from [GOVERNANCE CONTEXT].

Always return a final JSON bundle:
{
  "final_answer": "well-formatted response for the user",
  "agents_used": ["list of specialist agent names called"],
  "routing_decision": {"intent": "...", "risk_tier": "...", "persona": "...", "disclaimer_accepted": false},
  "confidence": 0.0,
  "governance_notices": ["any disclaimers or policy notices"],
  "disclaimer_required": false
}"""

async def setup():
    credential = DefaultAzureCredential()
    client = FoundryChatClient(
        project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        credential=credential,
    )
    agent = Agent(
        client=client, name="pf-orchestrator",
        instructions=ORCHESTRATOR_SYSTEM,
        tools=[contextualize_query, get_product_intelligence,
               check_compatibility, align_response, process_sample_request],
        default_options={{"store": False}},
    )
    return ResponsesHostServer(agent)

if __name__ == "__main__":
    asyncio.run(setup()).run()
'''

ORCH_REQUIREMENTS = """\
agent-framework>=1.2.0
agent-framework-foundry-hosting>=1.0.0a260507
azure-identity>=1.25.0
azure-monitor-opentelemetry>=1.6.0
requests>=2.32.0
"""

ORCH_DOCKERFILE = """\
FROM python:3.13-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY main.py .
COPY runtime-snapshot.json .
EXPOSE 8088
CMD ["python", "main.py"]
"""

(orch_dir / "main.py").write_text(ORCHESTRATOR_MAIN.strip(), encoding="utf-8")
(orch_dir / "requirements.txt").write_text(ORCH_REQUIREMENTS, encoding="utf-8")
(orch_dir / "Dockerfile").write_text(ORCH_DOCKERFILE, encoding="utf-8")
(orch_dir / "agent.yaml").write_text(
    f"kind: hosted\\nname: {ORCHESTRATOR_NAME}\\nprotocols:\\n"
    f"  - protocol: responses\\n    version: 1.0.0\\n"
    f"resources:\\n  cpu: \\\"2\\\"\\n  memory: 4Gi\\n"
    f"environment_variables:\\n"
    f"  - name: AZURE_AI_MODEL_DEPLOYMENT_NAME\\n    value: {PF_MODEL_FULL}\\n"
    f"  - name: APIM_GATEWAY_URL\\n    value: {APIM_GATEWAY_URL}\\n"
    f"  - name: APIM_SUBSCRIPTION_KEY\\n    value: {PF_SUB_KEY}\\n"
    f"  - name: OTEL_SERVICE_NAME\\n    value: {ORCHESTRATOR_NAME}\\n"
    f"  - name: ENABLE_INSTRUMENTATION\\n    value: \\\"true\\\"\\n"
    f"  - name: REGISTRY_SNAPSHOT_PATH\\n    value: runtime-snapshot.json\\n",
    encoding="utf-8"
)
utils.print_ok(f"Orchestrator source files written to: {orch_dir.name}/")

In [ ]:
# This cell is intentionally kept as a guard rail.
# Use the next build cell, which reads the current tag from ACR and increments right before build.
utils.print_warning("Build step moved: run the next cell to increment tag from live ACR state and build orchestrator image.")

In [ ]:
import time as _t

# ── 2️⃣  Build orchestrator image in ACR ─────────────────────────────────────
# Increment tag here, just before building, so deploy always uses the new image.
PREVIOUS_IMAGE_TAG = ORCHESTRATOR_IMAGE_TAG
ORCHESTRATOR_IMAGE_TAG = next_version_tag(PREVIOUS_IMAGE_TAG)
set_azd_env(ORCHESTRATOR_TAG_ENV_KEY, ORCHESTRATOR_IMAGE_TAG)
utils.print_info(f"{ORCHESTRATOR_NAME}: {PREVIOUS_IMAGE_TAG or '<none>'} -> {ORCHESTRATOR_IMAGE_TAG}")

utils.print_info(f"Building orchestrator image in ACR: {ORCHESTRATOR_NAME}:{ORCHESTRATOR_IMAGE_TAG}")
build_out = run(
    f"az acr build --registry {ACR_NAME} --resource-group {SPOKE_RG} "
    f"--image {ORCHESTRATOR_NAME}:{ORCHESTRATOR_IMAGE_TAG} "
    f"--file {AGENTS_BASE / ORCHESTRATOR_NAME}/Dockerfile "
    f"--no-logs {AGENTS_BASE / ORCHESTRATOR_NAME}/",
    "Orchestrator build queued", "Orchestrator build failed"
)
if not build_out.success:
    # Roll back tag on build failure.
    set_azd_env(ORCHESTRATOR_TAG_ENV_KEY, PREVIOUS_IMAGE_TAG)
    ORCHESTRATOR_IMAGE_TAG = PREVIOUS_IMAGE_TAG
    raise RuntimeError("Failed to queue ACR build for orchestrator")

utils.print_info("Polling ACR for orchestrator image presence...")
for attempt in range(30):
    tag_out = run(
        f"az acr repository show-tags --name {ACR_NAME} "
        f"--repository {ORCHESTRATOR_NAME} --output json",
        "", ""
    )
    if tag_out.success and tag_out.json_data and ORCHESTRATOR_IMAGE_TAG in tag_out.json_data:
        utils.print_ok(f"{ORCHESTRATOR_NAME}:{ORCHESTRATOR_IMAGE_TAG} verified in ACR")
        break
    utils.print_info(f"  Waiting for image... ({(attempt+1)*10}s)")
    _t.sleep(10)
else:
    raise RuntimeError(f"Image {ORCHESTRATOR_NAME}:{ORCHESTRATOR_IMAGE_TAG} not found in ACR after 5 min")

### 4️⃣ Orchestrator routing smoke tests

Three quick tests to verify dynamic routing before running the full scenario notebooks.

| Test | Expected routing |
|------|----------------|
| Recommendation (low risk) | contextualizer → product-intelligence → aligner |
| Compatibility without disclaimer | contextualizer → DISCLAIMER_GATE (blocked) |
| Out-of-domain | contextualizer → polite refusal |

In [ ]:
# ── 3️⃣  Orchestrator smoke tests through Foundry endpoint ─────────────────────
import json
import uuid

oc = project_client.get_openai_client(agent_name=ORCHESTRATOR_NAME)

def gov_message(persona: str, user_text: str, disclaimer_accepted: bool = True) -> str:
    return f"""[GOVERNANCE CONTEXT]
persona: {persona}
disclaimer_accepted: {'true' if disclaimer_accepted else 'false'}

{user_text}"""

def display_bundle(title: str, text: str):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    try:
        obj = json.loads(text)
        print(json.dumps(obj, indent=2))
    except Exception:
        print(text)

# ── Smoke test 1: Recommendation flow ─────────────────────────────────────────
print("\n🧪 SMOKE TEST 1: Recommendation flow")
resp1 = oc.responses.create(
    input=gov_message("external_customer", "I need a shampoo for my healthy 6-year-old golden retriever."),
    metadata={"conversation_id": str(uuid.uuid4())},
)
display_bundle("Recommendation (external_customer)", resp1.output_text)

# ── Smoke test 2: Compatibility without disclaimer ────────────────────────────
print("\n🧪 SMOKE TEST 2: Compatibility — disclaimer NOT accepted (should be gated)")
resp2 = oc.responses.create(
    input=gov_message("external_customer",
                      "Can I mix SynPet Clean Pro and SynPet Flea Guard?",
                      disclaimer_accepted=False),
    metadata={"conversation_id": str(uuid.uuid4())},
)
display_bundle("Compatibility — no disclaimer (should return DISCLAIMER_GATE)", resp2.output_text)
raw2 = resp2.output_text.lower()
if "disclaimer" in raw2 or "accept" in raw2 or "gate" in raw2:
    utils.print_ok("  ✅ Disclaimer gate activated correctly")
else:
    utils.print_warning("  ⚠️  Expected disclaimer gate — check orchestrator routing logic")

# ── Smoke test 3: Out-of-domain ────────────────────────────────────────────────
print("\n🧪 SMOKE TEST 3: Out-of-domain — should be refused politely")
resp3 = oc.responses.create(
    input=gov_message("external_customer", "What is the best recipe for spaghetti carbonara?"),
    metadata={"conversation_id": str(uuid.uuid4())},
)
display_bundle("Out-of-domain (should be refused)", resp3.output_text)
raw3 = resp3.output_text.lower()
if any(kw in raw3 for kw in ["not able","outside","only product","out of","cannot help"]):
    utils.print_ok("  ✅ Out-of-domain correctly refused")
else:
    utils.print_warning("  ⚠️  Expected domain refusal — check orchestrator system prompt")

# Persist orchestrator deployment flag to azd env.
set_azd_env("PF_ORCHESTRATOR_DEPLOYED", "true")

print()
utils.print_ok("✅ Orchestrator deployed and smoke tests complete. Proceed to scenario notebooks.")